# World Cup 2026 simulation with LightGBM

Train a match-result model on the gold feature table, then simulate the World Cup with probabilistic match sampling.

In [3]:
import numpy as np
import pandas as pd
import optuna
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder

from world_cup_features import (
    get_training_feature_columns,
    build_team_feature_store,
    prepare_match_row,
    prepare_world_cup_matches,
)
from world_cup_simulation import simulate_world_cup

ModuleNotFoundError: No module named 'football_analysis'

In [ ]:
gold_path = "football_analysis/data/gold_partidas_features/part-0.parquet"
fixtures_path = "football_analysis/model_notebooks/world_cup_2026_games.csv"

df = pd.read_parquet(gold_path)
fixtures = pd.read_csv(fixtures_path)

df["target"] = np.where(
    df["home_score"] > df["away_score"],
    "W",
    np.where(df["home_score"] < df["away_score"], "L", "D")
)

feature_columns = get_training_feature_columns(df)
X = df[feature_columns].fillna(0).astype("float32")
y = df["target"]

print(X.shape)
print(y.value_counts())

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

def objective(trial):
    params = {
        "objective": "multiclass",
        "num_class": len(le.classes_),
        "metric": "multi_logloss",
        "boosting_type": "gbdt",
        "verbosity": -1,
        "random_state": 42,
        "n_jobs": 1,
        "n_estimators": trial.suggest_int("n_estimators", 10, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 127),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 80),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }

    model = lgb.LGBMClassifier(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y_encoded, cv=cv, scoring="accuracy", n_jobs=1)
    return scores.mean()

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30, n_jobs=1)

print("Best score:", study.best_value)
print("Best params:", study.best_params)

In [ ]:
best_model = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=len(le.classes_),
    metric="multi_logloss",
    boosting_type="gbdt",
    verbosity=-1,
    random_state=42,
    n_jobs=1,
    **study.best_params,
)

best_model.fit(X, y_encoded)

importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": best_model.feature_importances_,
}).sort_values("importance", ascending=False)

importance_df.head(20)

In [ ]:
store = build_team_feature_store(df, feature_columns)

wc_feature_rows = prepare_world_cup_matches(fixtures, store)
wc_feature_rows[["MatchNumber", "HomeTeam", "AwayTeam", "home_team_has_history", "away_team_has_history"]].head(10)

In [ ]:
example_match = prepare_match_row("Brazil", "Germany", store).fillna(0).astype("float32")
example_proba = pd.DataFrame(
    [best_model.predict_proba(example_match)[0]],
    columns=[f"proba_{c}" for c in le.classes_],
)
example_proba

In [ ]:
simulation = simulate_world_cup(
    fixtures_df=fixtures,
    model=best_model,
    store=store,
    label_encoder=le,
    rng_seed=42,
)

print("Champion:", simulation["champion"])
simulation["standings"].head(50)

In [ ]:
simulation["third_place_ranking"].head(50)

In [ ]:
simulation["knockout_matches"]